# Final Activity: Improving DenseNet-121 for Cat and Dog Classification

**Name:** Romer Ecuben  

In [1]:
#cell 1
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
#cell 2
import tensorflow_datasets as tfds
train_ds, val_ds = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

In [ ]:
#cell 3
from tensorflow.keras.applications.densenet import preprocess_input

IMG_SIZE = (224, 224)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def preprocess_train(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = data_augmentation(image, training=True)
    image = preprocess_input(image)
    return image, label

def preprocess_val(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess_train).batch(32)
val_ds = val_ds.map(preprocess_val).batch(32)

In [ ]:
# cell 4 is deleted it caused double normalization (rescaling on already preprocessed images)

In [ ]:
#cell 5
base_model = tf.keras.applications.DenseNet121(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
#cell 6
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

In [ ]:
#cell 7
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#cell 8
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
# cell 9
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
#cell 10
plt.figure()
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

In [ ]:
#cell 11
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

y_true = np.concatenate([y.numpy() for x, y in val_ds], axis=0)

y_pred_probs = model.predict(val_ds)
y_pred = (y_pred_probs > 0.5).astype('int32').flatten()

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cat", "Dog"])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix (Blue Theme)")
plt.show()